DSCI 552 Homework 3

Name: Brynn Dafoe GitHub Username: brynndafoe02 USD ID: 3109-6692-10

In [70]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error, r2_score, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler

from itertools import combinations
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf

from pathlib import Path
import re

Question 1.b
- Keep datasets 1 and 2 in folders bending1 and bending 2
- Keep datasets 1, 2, and 3 in other folders as test data and other datasets as train data

In [71]:
# all folders -> lines 1-5 are headers, line 6 starts the data, separated by ","
path_to_AReM = Path("../data/AReM")
column_names = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]

training_dfs = {}
testing_dfs = {}

for data_dir in path_to_AReM.iterdir():
    if data_dir.is_dir(): # want to skip the .pdfs
        dir_name = data_dir.name
            
        for file in data_dir.iterdir():
            file_name = Path(file).stem
            file_num = int("".join(re.findall(r"\d", file_name))) # extracting the number to put in training or testing folder
            # ERROR TESTING !!!!
            # bad_count = 0
            # max_print = 5
            # with open(file, "r") as f:
            #     for i, line in enumerate(f):
            #         if i < 5:
            #             continue
            #         parts = line.strip().split(",")
            #         if len(parts) != len(column_names):
            #             print(f"BAD LINE -> {dir_name}, {file_name}, {line}")
            #             bad_count += 1
            # print(bad_count)
            # FOUND:
                # bending2 dataset 4 -> delimiter is " ", not "," -> fixing in the code
                # cycling dataset 9, 14 -> last line has trailing "," -> opened in TextEdit and removed the trailing ","

            key = f"{dir_name}_{file_name}" # bending1_dataset1 for example
            
            if (dir_name == "bending1") or (dir_name == "bending2"):
                # datasets 1, 2 = test. rest = train
                if dir_name == "bending1":
                    df = pd.read_csv(file, skiprows=5, names=column_names)
                    # first 5 rows are not data, adding the column names saved above
                else:
                    df = pd.read_csv(file, skiprows=5, names=column_names, sep=" ")
                    # this file has a different delimiter 
                
                if (file_num == 1) or (file_num == 2):
                    testing_dfs[key] = df
                else:
                    training_dfs[key] = df
            else: 
                # datasets 1, 2, 3, = test. rest = train
                df = pd.read_csv(file, skiprows=5, names=column_names)
                
                if (file_num == 1) or (file_num == 2) or (file_num == 3):
                    testing_dfs[key] = df
                else:
                    training_dfs[key] = df
          

Question 1.c
- Feature Extraction
- Classification of time series usually needs extracting features from them
- In this problem, we focus on time-domain features

Question 1.c.i
- Research what types of time-domain features are usually used in time series classification and list them (examples are minimum, maximum, mean, etc.)

1. Mean
2. Median
3. Mode
4. Min
5. Max
6. Range
7. Variance
8. Standard Deviation
9. Skewness (describes assymetry. 0 = symmetric. + = right-skewed. - = left-skewed)
10. Kurtosis (describes shape of a data distribution, quantifying how often extreme outliers occur)(tails)
11. Autocorrelation (measures relationship between a varibale and its lagged values / tells how much a data point is influenced by its own past values)
12. Zero Crossing Rate (rate at which signal changes signs)
14. Trend (a sequence of values recorded over time, used to analyize patterns / monitor changes / make forecasts)
15. Seasonality (recurring / regular patterns at a set interval)

Question 1.c.ii
- Extract the time-domain features:
- - minimum, maximum, mean, median, standard deviation, first quartile, third quartile
- for all of the 6 time series in each instance (the column names saved earlier)
- Free to normalize / standardize features or use them directly (can experiment to see if it makes a difference)
- New dataset will look like:
- - Instance (1 -> 88), Min1, Max1, Mean1, Median1, ... 1st quart6, 3rd quart6
  - (1st quart6 = first quartile of the SIXTH time series in each of the 88 instances)

For each dataset in each folder: -> 42 columns for each new dataset
1. avg_rss12 -> min, max, mean, median, std dev, 1Q, 3Q
2. var_rss12 -> min, max, mean, median, std dev, 1Q, 3Q
3. avg_rss13 -> min, max, mean, median, std dev, 1Q, 3Q
4. var_rss13 -> min, max, mean, median, std dev, 1Q, 3Q
5. avg_rss23 -> min, max, mean, median, std dev, 1Q, 3Q
6. var_rss23 -> min, max, mean, median, std dev, 1Q, 3Q

In [72]:
columns_six = column_names[1:] # not using "time"

training_features = [] # will by 69 rows x 42 columns
# want it to look like [{dirfilename : name, min1 : x, max1 : x, ...}, {}, ...]

for dirfile1, its_df1 in training_dfs.items():
    a_instance = {}
    a_instance["dirfile_name"] = dirfile1 # to keep track of where the numbers are coming from

    for i, a_column in enumerate(columns_six, start=1):
        a_instance[f"min_{i}"] = its_df1[a_column].min()
        a_instance[f"max_{i}"] = its_df1[a_column].max()
        a_instance[f"mean_{i}"] = its_df1[a_column].mean()
        a_instance[f"median_{i}"] = its_df1[a_column].median()
        a_instance[f"std_dev_{i}"] = its_df1[a_column].std()
        a_instance[f"first_quart_{i}"] = its_df1[a_column].quantile(0.25)
        a_instance[f"third_quart_{i}"] = its_df1[a_column].quantile(0.75)

    training_features.append(a_instance)

In [73]:
testing_features = [] # will by 19 rows x 42 columns

for dirfile2, its_df2 in testing_dfs.items():
    b_instance = {}
    b_instance["dirfile_name"] = dirfile2

    for j, b_column in enumerate(columns_six, start=1):
        b_instance[f"min_{j}"] = its_df2[b_column].min()
        b_instance[f"max_{j}"] = its_df2[b_column].max()
        b_instance[f"mean_{j}"] = its_df2[b_column].mean()
        b_instance[f"median_{j}"] = its_df2[b_column].median()
        b_instance[f"std_dev_{j}"] = its_df2[b_column].std()
        b_instance[f"first_quart_{j}"] = its_df2[b_column].quantile(0.25)
        b_instance[f"third_quart_{j}"] = its_df2[b_column].quantile(0.75)

    testing_features.append(b_instance)